# Extensions: d=5 validation, baselines on real panels, split-policy exhibit 
Three pre-registered extensions.

**Part 1 (d = 5, known order).** DGPs among five variables with nuisance
dimensions -- o1: x1 + tanh(x2) - 0.5 x3 + 0.3 x4 - tanh(x5);
o2: x1 x2 + tanh(x3) + x4; o3: x1 x2 x3 + x4 + tanh(x5). Dependence:
independent; corr(x1, x3) in {0.5, 0.9}. sigma = 0.5, n = 20,000, R = 10,
K = 3, D = 4 (121 features at k = 3). Checks: overselection on o1/o2 within
the binomial 1%-tail of Binomial(60, 0.05); power on o3 at independent and
pair-0.5 >= 0.9. The pair-0.9 o3 cell is recorded as an observation: the
nuisance dimensions dilute the remaining gap to ~0.005 of outcome variance,
near but likely above the sigma = 0.5 floor.

**Part 2 (baselines on the six V-Dem sweeps).** NID (numpy MLP; threshold
TRANSFERRED from the synthetic o1 calibration stored in
results/baselines_validation/nid_tau.json -- a transferred calibration,
labeled as such), the pairwise-capped baseline (hierNet via rpy2 with the
flagged all-pairs 1SE-lasso fallback), and CV-1SE with COUNTRY-BLOCKED
folds (fold assignment over countries, so the classical rule gets the same
leakage protection as the sweep). PRE-REGISTERED prediction, implementation-conditional: under
rpy2-hierNet at its CV-selected penalty, the baseline reports interactions
(k_hat = 2) on a majority of the six sweeps, by its 76% false-report rate
on purely additive synthetic data; under the conservative 1SE-lasso
fallback the same synthetic evidence predicts k_hat = 1 throughout (the
fallback was clean on synthetic additive cells). The check names the
implementation that ran and evaluates the matching branch. NID under
transferred calibration carries no strong prior.

**Part 3 (split-policy exhibit).** The six sweeps rerun with naive ROW
splits. PRE-REGISTERED prediction (observation, not check): row-split
k_hat >= blocked k_hat with at least one strict inequality -- within-country
leakage lets richer classes memorize country level shifts.

Acceptance checks are integrity and Part-1 ground-truth only; Parts 2-3 on
real data are findings. Outputs to
`MyDrive/ORDER_SWEEP/results/extensions/`.


In [ ]:
# Cell 1 -- Mount Drive; locate data and the stored NID calibration
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ORDER_SWEEP'
OUT = os.path.join(BASE, 'results', 'extensions')
os.makedirs(OUT, exist_ok=True)
DATA = os.path.join(BASE, 'data', 'HDL_merged_notdev_selected.csv')
TAU_PATH = os.path.join(BASE, 'results', 'baselines_validation', 'nid_tau.json')
assert os.path.exists(DATA), f"missing {DATA}"
assert os.path.exists(TAU_PATH), f"missing {TAU_PATH} (synthetic NID calibration)"
print('output folder:', OUT)


In [ ]:
# Cell 2 -- Machinery: blocked selector, blocked CV-1SE, NID, pairwise baseline
import numpy as np, json, csv, time, hashlib, zlib, sys, platform, inspect
import scipy
from itertools import product as iproduct
from scipy import stats

def monomial_exps(d, D, max_active):
    out = []
    for combo in iproduct(range(D + 1), repeat=d):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def poly_design(X, D, max_active):
    d = X.shape[1]
    exps = monomial_exps(d, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0] * d))

def split_indices(n, split_seed, train_frac, blocks):
    if blocks is None:
        idx = np.random.default_rng(split_seed).permutation(n)
        return idx[: int(train_frac * n)], idx[int(train_frac * n):]
    uniq = np.unique(blocks)
    perm = np.random.default_rng(split_seed).permutation(len(uniq))
    tr_b = set(uniq[perm[: int(train_frac * len(uniq))]])
    tr_mask = np.fromiter((b in tr_b for b in blocks), bool, count=n)
    return np.where(tr_mask)[0], np.where(~tr_mask)[0]

def select_order(X, h, K=3, S=10, alpha=0.05, D=4, train_frac=0.75, blocks=None):
    """Frozen fixed-sequence selector with blocked-split policy (identical
    to the validated implementation of the panel-application artifact)."""
    n = X.shape[0]
    designs = {k: poly_design(X, D, k) for k in range(1, K + 1)}
    r2, ntr_rows, nte_rows = {}, [], []
    for s in range(S):
        tr, te = split_indices(n, s, train_frac, blocks)
        out = {}
        for k in range(1, K + 1):
            Phi0, ci = designs[k]
            mu = Phi0[tr].mean(0); sd = Phi0[tr].std(0); sd[sd == 0] = 1.0
            Phi = (Phi0 - mu) / sd
            Phi[:, ci] = 1.0
            hm = h[tr].mean()
            beta, *_ = np.linalg.lstsq(Phi[tr], h[tr] - hm, rcond=None)
            resid = (h[te] - hm) - Phi[te] @ beta
            out[k] = 1.0 - float((resid @ resid) / np.sum((h[te] - h[te].mean()) ** 2))
        r2[s] = out; ntr_rows.append(len(tr)); nte_rows.append(len(te))
    if blocks is None:
        ratio = np.mean(nte_rows) / np.mean(ntr_rows)
    else:
        n_b = len(np.unique(blocks)); n_tr_b = int(train_frac * n_b)
        ratio = (n_b - n_tr_b) / n_tr_b
    corr = 1.0 / S + ratio
    p_feat = {k: designs[k][0].shape[1] for k in range(1, K + 1)}
    one_minus_r2K = float(np.mean([1.0 - r2[s][K] for s in range(S)]))
    n_tr_mean = float(np.mean(ntr_rows))
    stat = {}
    for k in range(1, K):
        g = np.array([r2[s][K] - r2[s][k] for s in range(S)])
        m = g.mean(); v = g.var(ddof=1) * corr
        opt = (p_feat[K] - p_feat[k]) * one_minus_r2K / n_tr_mean
        if v > 0:
            t = m / np.sqrt(v); p = 1.0 - stats.t.cdf(t, df=S - 1)
            ub = m + stats.t.ppf(1 - alpha, df=S - 1) * np.sqrt(v) + opt
        else:
            t = np.inf if m > 0 else (-np.inf if m < 0 else 0.0)
            p = 0.0 if m > 0 else 1.0
            ub = m + opt
        stat[k] = {"mean": float(m), "p": float(p), "ub": float(ub),
                   "pi": float((g > 0).mean())}
    khat, ub_cert = K, None
    for k in range(1, K):
        if stat[k]["p"] > alpha:
            khat, ub_cert = k, stat[k]["ub"]
            break
    return khat, stat, ub_cert

def cv1se_khat(X, h, K=3, D=4, folds=5, seed=0, blocks=None):
    """CV-1SE with optional BLOCKED folds (fold assignment over blocks)."""
    n = X.shape[0]
    rng = np.random.default_rng(seed)
    if blocks is None:
        idx = rng.permutation(n)
        fold_id = np.empty(n, dtype=int); fold_id[idx] = np.arange(n) % folds
    else:
        uniq = np.unique(blocks)
        bfold = {b: f for b, f in zip(uniq[rng.permutation(len(uniq))],
                                      np.arange(len(uniq)) % folds)}
        fold_id = np.fromiter((bfold[b] for b in blocks), int, count=n)
    errs = {k: [] for k in range(1, K + 1)}
    for k in range(1, K + 1):
        Phi0, ci = poly_design(X, D, k)
        for f in range(folds):
            tr, te = fold_id != f, fold_id == f
            mu = Phi0[tr].mean(0); sd = Phi0[tr].std(0); sd[sd == 0] = 1.0
            P = (Phi0 - mu) / sd; P[:, ci] = 1.0
            hm = h[tr].mean()
            beta, *_ = np.linalg.lstsq(P[tr], h[tr] - hm, rcond=None)
            resid = (h[te] - hm) - P[te] @ beta
            errs[k].append(float(np.mean(resid ** 2)))
    means = {k: float(np.mean(v)) for k, v in errs.items()}
    ses = {k: float(np.std(v, ddof=1) / np.sqrt(folds)) for k, v in errs.items()}
    kbest = min(means, key=means.get)
    return min(k for k in range(1, K + 1) if means[k] <= means[kbest] + ses[kbest])

def mlp_train(X, h, hidden=(64, 32), epochs=300, lr=1e-2, seed=0, l1_w1=1e-4):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    sizes = [d] + list(hidden) + [1]
    W = [rng.standard_normal((a, b)) * np.sqrt(2.0 / a) for a, b in zip(sizes, sizes[1:])]
    b = [np.zeros(s) for s in sizes[1:]]
    mW = [np.zeros_like(w) for w in W]; vW = [np.zeros_like(w) for w in W]
    mB = [np.zeros_like(x) for x in b]; vB = [np.zeros_like(x) for x in b]
    b1, b2, eps = 0.9, 0.999, 1e-8
    hs = (h - h.mean()) / (h.std() + 1e-12)
    for t in range(1, epochs + 1):
        acts = [X]
        for li, (w, bb) in enumerate(zip(W, b)):
            z = acts[-1] @ w + bb
            acts.append(np.maximum(z, 0.0) if li < len(W) - 1 else z)
        grad = ((2.0 / n) * (acts[-1][:, 0] - hs))[:, None]
        gW, gB = [None] * len(W), [None] * len(b)
        for li in range(len(W) - 1, -1, -1):
            gW[li] = acts[li].T @ grad; gB[li] = grad.sum(0)
            if li > 0: grad = (grad @ W[li].T) * (acts[li] > 0)
        gW[0] = gW[0] + l1_w1 * np.sign(W[0])
        for li in range(len(W)):
            for g, w, mm, vv in [(gW[li], W[li], mW[li], vW[li]),
                                 (gB[li], b[li], mB[li], vB[li])]:
                mm *= b1; mm += (1 - b1) * g
                vv *= b2; vv += (1 - b2) * g * g
                w -= lr * (mm / (1 - b1**t)) / (np.sqrt(vv / (1 - b2**t)) + eps)
    return W

def nid_khat(X, h, tau, seed=0):
    W = mlp_train(X, h, seed=seed)
    W1 = np.abs(W[0]); mu = np.abs(W[1])
    for w in W[2:]: mu = mu @ np.abs(w)
    mu = mu[:, 0]
    d = X.shape[1]
    orders = [1]
    from itertools import combinations
    for r in [2, 3]:
        for I in combinations(range(d), r):
            if float(np.sum(np.min(W1[list(I), :], axis=0) * mu)) > tau:
                orders.append(r)
    return max(orders)

PAIRWISE_IMPL = None
try:
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri
    numpy2ri.activate()
    try:
        ro.r("suppressMessages(library(hierNet))")
    except Exception:
        ro.r("install.packages('hierNet', repos='https://cloud.r-project.org', quiet=TRUE)")
        ro.r("suppressMessages(library(hierNet))")
    PAIRWISE_IMPL = "rpy2-hierNet"
    def pairwise_khat(X, h, seed=0):
        sub = np.random.default_rng(seed).permutation(len(X))[:4000]
        ro.globalenv["x"] = X[sub]; ro.globalenv["y"] = h[sub]
        ro.r("x <- as.matrix(x); y <- as.numeric(y)")
        ro.r("fit <- hierNet.path(x, y, trace=0)")
        ro.r("cv <- hierNet.cv(fit, x, y, nfolds=5, trace=0)")
        nnz = ro.r("f2 <- hierNet(x, y, lam=cv$lamhat); sum(abs(f2$th) > 1e-8)")[0]
        return 2 if nnz > 0 else 1
except Exception as e:
    PAIRWISE_IMPL = "allpairs-lasso-1se (NOT hierNet)"
    from sklearn.linear_model import LassoCV, Lasso
    from itertools import combinations as _comb
    def pairwise_khat(X, h, seed=0):
        pairs = [X[:, i] * X[:, j] for i, j in _comb(range(X.shape[1]), 2)]
        Z = np.column_stack([X] + pairs)
        Z = (Z - Z.mean(0)) / np.where(Z.std(0) == 0, 1.0, Z.std(0))
        hs = (h - h.mean()) / (h.std() + 1e-12)
        cv = LassoCV(cv=5, alphas=40, random_state=seed).fit(Z, hs)
        mse = cv.mse_path_.mean(axis=1)
        se = cv.mse_path_.std(axis=1, ddof=1) / np.sqrt(cv.mse_path_.shape[1])
        best = int(np.argmin(mse))
        alpha_1se = float(cv.alphas_[np.argmax(mse <= mse[best] + se[best])])
        fit = Lasso(alpha=alpha_1se).fit(Z, hs)
        return 2 if np.any(np.abs(fit.coef_[X.shape[1]:]) > 1e-8) else 1
print("machinery ready; pairwise implementation:", PAIRWISE_IMPL)

def _fn_repr(f):
    try:
        return inspect.getsource(f)
    except OSError:
        c = f.__code__
        return repr((c.co_code, c.co_consts, c.co_names, c.co_varnames))
SEED_SCHEME = "crc32-full-v3"
MACH_SRC = "\n\n".join(_fn_repr(f) for f in
    [monomial_exps, poly_design, split_indices, select_order, cv1se_khat,
     mlp_train, nid_khat, pairwise_khat])


In [ ]:
# Cell 3 -- Part 1: d = 5 validation with known order
EXP1 = "extensions_d5"
D5_DGPS = {"o1_5": 1, "o2_5": 2, "o3_5": 3}
D5_DEPS = ["indep", "pair0.5", "pair0.9"]
N5, R5, SIG5 = 20_000, 10, 0.5
_cfg1 = {"N": N5, "R": R5, "SIGMA": SIG5, "DGPS": list(D5_DGPS), "DEPS": D5_DEPS,
         "K": 3, "D": 4, "S": 10, "ALPHA": 0.05, "SEED_SCHEME": SEED_SCHEME}
SHA1 = hashlib.sha256(MACH_SRC.encode() + json.dumps(_cfg1, sort_keys=True).encode()).hexdigest()
print("d5 provenance:", SHA1[:16], "...")

def make_d5(dgp, dep, rep):
    seed = (30_000 + zlib.crc32(f"d5|{dgp}|{dep}".encode()) + rep * 977) % 2**32
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal((N5, 5))
    if dep.startswith("pair"):
        rho = float(dep[4:])
        Z[:, 2] = rho * Z[:, 0] + np.sqrt(1 - rho**2) * Z[:, 2]
    x1, x2, x3, x4, x5 = Z.T
    f = {"o1_5": x1 + np.tanh(x2) - 0.5 * x3 + 0.3 * x4 - np.tanh(x5),
         "o2_5": x1 * x2 + np.tanh(x3) + x4,
         "o3_5": x1 * x2 * x3 + x4 + np.tanh(x5)}[dgp]
    return Z, f + SIG5 * rng.standard_normal(N5)

t0 = time.time()
rows1 = []
for dgp, tk in D5_DGPS.items():
    for dep in D5_DEPS:
        for rep in range(R5):
            X, h = make_d5(dgp, dep, rep)
            khat, stat, ub = select_order(X, h)
            rows1.append({"experiment": EXP1, "dgp": dgp, "dependence": dep,
                          "rep": rep, "true_order": tk, "khat": khat,
                          "rem1_p": stat[1]["p"], "rem2_p": stat[2]["p"],
                          "rem2_mean": stat[2]["mean"],
                          "ub_cert": "" if ub is None else ub})
        print(f"{dgp} {dep} done ({time.time()-t0:4.0f}s)", flush=True)
with open(os.path.join(OUT, "per_seed_d5.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows1[0].keys())); w.writeheader(); w.writerows(rows1)

nulls = [r for r in rows1 if r["dgp"] in ("o1_5", "o2_5")]
over = sum(r["khat"] > r["true_order"] for r in nulls)
m_crit, tail = 0, 1.0
while tail >= 0.01:
    m_crit += 1
    tail = 1.0 - stats.binom.cdf(m_crit - 1, len(nulls), 0.05)
p3_rows = [r for r in rows1
           if r["dgp"] == "o3_5" and r["dependence"] in ("indep", "pair0.5")]
p3 = sum(r["khat"] == 3 for r in p3_rows)
c1_checks = [(f"d5 calibration: {over}/{len(nulls)} overselections < binomial 1%-tail count {m_crit}", over < m_crit),
             (f"d5 power (o3, indep + pair-0.5): {p3}/{len(p3_rows)} >= 0.9", p3 / len(p3_rows) >= 0.9)]
s1 = []
for name, ok in c1_checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    s1.append(line); print(line)
p9 = [r for r in rows1 if r["dgp"] == "o3_5" and r["dependence"] == "pair0.9"]
s1.append(f"OBS   d5 o3 at pair-0.9: khat distribution "
          + str({k: sum(r['khat'] == k for r in p9) for k in [1, 2, 3]})
          + f", mean rem2 {np.mean([r['rem2_mean'] for r in p9]):.5f} (diluted gap near the floor)")
print(s1[-1])
with open(os.path.join(OUT, "check_d5.txt"), "w") as f:
    f.write("\n".join(s1) + "\n")
with open(os.path.join(OUT, "metadata_d5.json"), "w") as f:
    json.dump({"experiment": EXP1, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "config": _cfg1, "code_sha256": SHA1, "numpy": np.__version__,
               "scipy": scipy.__version__, "python": sys.version}, f, indent=2)
print("wrote per_seed_d5.csv, check_d5.txt, metadata_d5.json")


In [ ]:
# Cell 4 -- Parts 2-3: baselines on the six V-Dem sweeps + split-policy exhibit
import pandas as pd
EXP2 = "extensions_vdem"
TRIPLES = {"navco":  ["navco_nonviol", "v2csprtcpt", "v2clrspct"],
           "party":  ["v2xps_party", "v2csprtcpt", "v2clrspct"],
           "antimv": ["v2csantimv", "v2csprtcpt", "v2clrspct"]}
OUTCOMES = ["upturn", "downturn"]
TAU = json.load(open(TAU_PATH))["tau"]
_cfg2 = {"TRIPLES": TRIPLES, "OUTCOMES": OUTCOMES, "nid_tau_transferred": TAU,
         "pairwise_impl": PAIRWISE_IMPL, "cv1se_blocked_folds": True,
         "S": 10, "ALPHA": 0.05, "K": 3, "D": 4, "SEED_SCHEME": SEED_SCHEME}
SHA2 = hashlib.sha256(MACH_SRC.encode() + json.dumps(_cfg2, sort_keys=True).encode()).hexdigest()
print(f"vdem-extensions provenance: {SHA2[:16]}...  transferred NID tau = {TAU:.5f}")

df = pd.read_csv(DATA)
t0 = time.time()
rows2 = []
for outc in OUTCOMES:
    for tname, cols in TRIPLES.items():
        sub = df[cols + [outc, "country_id"]].dropna()
        X = sub[cols].to_numpy(float); h = sub[outc].to_numpy(float)
        blocks = sub["country_id"].to_numpy()
        dh = hashlib.sha256(X.tobytes() + h.tobytes()).hexdigest()
        kb, stat_b, ub_b = select_order(X, h, blocks=blocks)
        kr, stat_r, ub_r = select_order(X, h, blocks=None)
        Xs = (X - X.mean(0)) / np.where(X.std(0) == 0, 1.0, X.std(0))
        k_nid = nid_khat(Xs, h, TAU, seed=0)
        k_pw = pairwise_khat(X, h, seed=0)
        k_cv = cv1se_khat(X, h, seed=0, blocks=blocks)
        rows2.append({"experiment": EXP2, "outcome": outc, "triple": tname,
                      "n": len(sub), "data_hash": dh,
                      "khat_blocked": kb, "ub_blocked": "" if ub_b is None else ub_b,
                      "khat_row": kr, "ub_row": "" if ub_r is None else ub_r,
                      "rem1_row_p": stat_r[1]["p"], "rem2_row_p": stat_r[2]["p"],
                      "khat_nid": k_nid, "khat_pairwise": k_pw, "khat_cv1se_blocked": k_cv})
        print(f"{outc:9s} {tname:7s} blocked={kb} row={kr} nid={k_nid} "
              f"pw={k_pw} cv={k_cv} ({time.time()-t0:4.0f}s)", flush=True)
with open(os.path.join(OUT, "per_seed_vdem_ext.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows2[0].keys())); w.writeheader(); w.writerows(rows2)
with open(os.path.join(OUT, "metadata_vdem_ext.json"), "w") as f:
    json.dump({"experiment": EXP2, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "config": _cfg2, "code_sha256": SHA2, "numpy": np.__version__,
               "scipy": scipy.__version__, "pandas": pd.__version__,
               "python": sys.version, "platform": platform.platform()}, f, indent=2)
print("wrote per_seed_vdem_ext.csv, metadata_vdem_ext.json")


In [ ]:
# Cell 5 -- Verification and findings report
import csv as _csv
rows2 = list(_csv.DictReader(open(os.path.join(OUT, "per_seed_vdem_ext.csv"))))
assert all(r["experiment"] == "extensions_vdem" for r in rows2), "stamp mismatch"
checks, story = [], []
checks.append((f"completeness: {len(rows2)} sweeps (expect 6), data_hash on every row",
               len(rows2) == 6 and all(len(r["data_hash"]) == 64 for r in rows2)))
checks.append((f"pairwise implementation recorded: {PAIRWISE_IMPL}", True))
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)

print(f"\n{'outcome':9s}{'triple':8s}{'blocked':>8s}{'row':>5s}{'NID':>5s}{'pairw':>6s}{'CV-1SE(b)':>10s}")
for r in rows2:
    print(f"{r['outcome']:9s}{r['triple']:8s}{r['khat_blocked']:>8s}{r['khat_row']:>5s}"
          f"{r['khat_nid']:>5s}{r['khat_pairwise']:>6s}{r['khat_cv1se_blocked']:>10s}")
    story.append(f"OBS   {r['outcome']}/{r['triple']}: blocked={r['khat_blocked']} "
                 f"row={r['khat_row']} nid={r['khat_nid']} pairwise={r['khat_pairwise']} "
                 f"cv1se_blocked={r['khat_cv1se_blocked']}")
pw2 = sum(int(r["khat_pairwise"]) >= 2 for r in rows2)
if PAIRWISE_IMPL == "rpy2-hierNet":
    verdict = "CONFIRMED" if pw2 >= 4 else "NOT confirmed"
    story.append(f"OBS   pre-registered prediction (hierNet-at-lamhat reports interactions on a "
                 f"majority): {pw2}/6 sweeps -- {verdict}")
else:
    verdict = "CONFIRMED" if pw2 == 0 else "NOT confirmed"
    story.append(f"OBS   pre-registered prediction (1SE fallback stays at k=1 throughout): "
                 f"{6-pw2}/6 sweeps at k=1 -- {verdict} [fallback branch]")
strict = sum(int(r["khat_row"]) > int(r["khat_blocked"]) for r in rows2)
geq = all(int(r["khat_row"]) >= int(r["khat_blocked"]) for r in rows2)
story.append(f"OBS   pre-registered prediction (row >= blocked, >=1 strict): "
             f"geq={'yes' if geq else 'NO'}, strict inequalities={strict}/6 -- "
             f"{'CONFIRMED' if geq and strict >= 1 else 'NOT confirmed'}")
for line in story[len(checks):]:
    print(line)
with open(os.path.join(OUT, "check_vdem_ext.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check_vdem_ext.txt")
